# STAR-RIS RSMA — Scalability theo N: **Lợi thế phân rã của MADDPG**
## Notebook bổ trợ (củng cố lý thuyết MADDPG/CTDE)

**Mục đích:** Chứng minh điều mà so sánh tại N=32 KHÔNG cho thấy — *lợi thế của MADDPG so với đơn-agent
(TD3/DDPG) **tăng theo kích thước không gian hành động**, tức theo số phần tử STAR-RIS **N**.*

**Cơ sở lý thuyết (CTDE / Lowe et al. 2017):** hành động đơn-agent có số chiều ≈ `2N + N + (K+1) + K`
(N=16→57, N=32→105, N=64→201). Khi N tăng, một policy đơn-agent phải khám phá không gian hành động
khổng lồ → khó. MADDPG **phân rã** thành 3 actor chuyên biệt `[BS power (9), RIS-R (2N), RIS-T (N)]`,
nên độ khó mỗi actor tăng chậm hơn → **MADDPG tách top khi N lớn**.

**📌 Kết quả mong đợi (để củng cố bài báo):**
- `sum-rate vs N`: đường MADDPG **vượt dần** TD3/DDPG khi N tăng.
- `Decomposition gain = SR(MADDPG) − SR(best single-agent)` **tăng đơn điệu theo N** (≥0 và lớn dần).
- Nếu xu hướng đúng → đây là **bằng chứng định lượng** cho việc chọn MADDPG (không chỉ "bằng TD3 ở N=32").

> ⚠️ **Compute:** đây là experiment huấn-luyện-lại-mỗi-N. Cấu hình mặc định (MADDPG+TD3, 3 seeds,
> N∈{16,32,64}, 1500 ep) nhắm vừa **1 session Kaggle (<12h)**. Chạy RIÊNG, không gộp với notebook chính.


### 1. Kiểm tra phần cứng & cài thư viện

In [ ]:
import torch
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
!pip install gymnasium pyyaml tqdm pandas matplotlib tensorboard

### 2. Đường dẫn dự án & import

In [ ]:
import os, sys, copy, time
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from IPython.display import Image, display

# ⚠️ Sửa cho khớp tên dataset bạn upload lên Kaggle:
PROJECT_ROOT = "/kaggle/input/datasets/duythanhb1909984/star-ris-rsma-maddpg-v14"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from experiments.train import train_maddpg, train_single_agent
from experiments.evaluate import _eval_multi_seed
from utils.plotting import plot_metric_vs_x
from utils import confidence_interval
print("Import OK")

### 3. Cấu hình v18 + tham số scalability

**📌 Output mong đợi:** in ra danh sách N, số seed/episode, và **ước lượng thời gian**.

> Chỉnh `N_LIST`, `SCAL_SEEDS`, `SCAL_EPISODES`, `SCAL_ALGOS` tùy ngân sách. Thêm DDPG hoặc N=96 nếu còn giờ.


In [ ]:
config_path = os.path.join(PROJECT_ROOT, "config", "config.yaml")
with open(config_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

# ----- Đồng bộ tham số v18 (freeze λ + cân bằng QoS) để công bằng với notebook chính -----
cfg["env"]["qos_lambda_freeze_fraction"] = 0.55
cfg["env"]["qos_lambda_max"] = 13.0
cfg["env"]["qos_target_satisfaction"] = 0.50
cfg["env"]["qos_lambda_increase"] = 1.02
cfg["env"]["phase_residual_scale"] = 0.25

# ----- Tham số scalability (điều chỉnh theo ngân sách Kaggle) -----
N_LIST        = [16, 32, 64]              # số phần tử STAR-RIS để quét
SCAL_ALGOS    = ["maddpg", "td3"]         # head-to-head; thêm "ddpg" nếu còn giờ
SCAL_SEEDS    = [1000, 2000, 3000]        # 3 seeds đủ thấy xu hướng
SCAL_EPISODES = 1500                      # đủ hội tụ (curve phẳng ~ep 1000)

LABEL = {"maddpg": "MADDPG", "td3": "TD3", "ddpg": "DDPG"}

out_root = "/kaggle/working/"
fig_dir  = os.path.join(out_root, "figures")
tab_dir  = os.path.join(out_root, "tables")
log_dir  = os.path.join(out_root, "logs_scal")
ckpt_dir = os.path.join(out_root, "checkpoints_scal")
for d in (fig_dir, tab_dir):
    os.makedirs(d, exist_ok=True)

# Ước lượng thời gian (thô, theo nhịp ~ N=32, 2000ep: MADDPG~38ph, TD3~14ph/seed)
_base = {"maddpg": 38.0, "td3": 14.0, "ddpg": 14.0}
est_min = 0.0
for N in N_LIST:
    scale_N = (N / 32.0)            # chi phí ~ tuyến tính theo N (action-dim)
    for a in SCAL_ALGOS:
        est_min += _base[a] * (SCAL_EPISODES / 2000.0) * max(scale_N, 0.5) * len(SCAL_SEEDS)
print("="*60)
print("  SCALABILITY SWEEP — cấu hình")
print("="*60)
print(f"  N_LIST          = {N_LIST}")
print(f"  Algos           = {SCAL_ALGOS}")
print(f"  Seeds           = {SCAL_SEEDS}")
print(f"  Episodes/seed   = {SCAL_EPISODES}")
print(f"  Số lần train    = {len(N_LIST)*len(SCAL_ALGOS)*len(SCAL_SEEDS)}")
print(f"  λ: freeze={cfg['env']['qos_lambda_freeze_fraction']}, max={cfg['env']['qos_lambda_max']}, "
      f"residual_scale={cfg['env']['phase_residual_scale']}")
print(f"  ⏱️  Ước lượng    ~ {est_min/60.0:.1f} giờ (T4). Nếu > 11h: bỏ bớt N hoặc seed!")
print("="*60)

### 4. Huấn luyện lại tại mỗi N (MADDPG vs single-agent)

**📌 Output mong đợi:** với mỗi `N`, tqdm bar cho từng (algo × seed). Mất nhiều giờ — đây là cell nặng nhất.

> Mỗi N: deep-copy config, đặt `num_ris_elements=N`, huấn luyện từ đầu (agent build theo `env.spec()`),
> rồi đánh giá đa-seed với λ huấn luyện-cuối. Kết quả lưu vào `scal_results`.


In [ ]:
scal_results = {}      # scal_results[N][label] = {"sr_mean","sr_ci","qos_mean","qos_ci","srs"}
rows = []

for N in N_LIST:
    cfg_N = copy.deepcopy(cfg)
    cfg_N["env"]["num_ris_elements"] = int(N)
    cfg_N["training"]["total_episodes"] = int(SCAL_EPISODES)
    print(f"\n{'#'*64}\n#  N = {N}  (single-agent action-dim ~ {3*N + 9})\n{'#'*64}")
    per_algo = {}
    for algo in SCAL_ALGOS:
        label = LABEL[algo]
        srs, qoss = [], []
        for s in SCAL_SEEDS:
            run = f"scal_{algo}_N{N}_s{s}"
            print(f"\n--- Train {label} | N={N} | seed={s} ---")
            if algo == "maddpg":
                info = train_maddpg(cfg_N, log_dir=log_dir, ckpt_dir=ckpt_dir,
                                    seed_override=s, run_name=run)
            else:
                info = train_single_agent(cfg_N, kind=algo, log_dir=log_dir, ckpt_dir=ckpt_dir,
                                          seed_override=s, run_name=run)
            lam = info.get("trained_qos_lambda")
            m = _eval_multi_seed(info["agent"], algo, info["obs_norm"], cfg_N,
                                 cfg_N["evaluation"]["seeds"], qos_lambda=lam)
            srs.append(m["sum_rate_mean"]); qoss.append(m["qos_mean"])
        sr_m, sr_ci, _ = confidence_interval(np.array(srs))
        q_m,  q_ci, _  = confidence_interval(np.array(qoss))
        per_algo[label] = {"sr_mean": sr_m, "sr_ci": sr_ci,
                           "qos_mean": q_m, "qos_ci": q_ci, "srs": srs}
        rows.append({"N": N, "Algorithm": label,
                     "SumRate": sr_m, "SumRate_CI": sr_ci,
                     "QoS": q_m, "QoS_CI": q_ci, "N_seeds": len(SCAL_SEEDS)})
        print(f"  => {label} N={N}: SR={sr_m:.3f}±{sr_ci:.3f}, QoS={q_m:.3f}±{q_ci:.3f}")
    scal_results[N] = per_algo

df_scal = pd.DataFrame(rows)
df_scal.to_csv(os.path.join(tab_dir, "scalability_vs_N.csv"), index=False)
print("\n--- Bảng scalability (scalability_vs_N.csv) ---")
display(df_scal)

### 5. Đồ thị & phân tích lợi thế phân rã

**📌 Output mong đợi:** 3 biểu đồ — (1) SR vs N, (2) QoS vs N, (3) **Decomposition gain vs N**.

**🎯 Bằng chứng cần có:** đường *gain* (MADDPG − best single-agent) **đi lên theo N**.
Nếu đúng ⇒ viết được: *"lợi thế của MADDPG so với đơn-agent tăng theo số phần tử RIS, khẳng định giá trị
của kiến trúc phân rã CTDE khi không gian hành động lớn."*


In [ ]:
labels = [LABEL[a] for a in SCAL_ALGOS]

# (1) Sum-rate vs N
sr_curves  = {lab: {"mean": [scal_results[N][lab]["sr_mean"] for N in N_LIST],
                    "ci":   [scal_results[N][lab]["sr_ci"]   for N in N_LIST]} for lab in labels}
plot_metric_vs_x(N_LIST, sr_curves, xlabel="STAR-RIS elements $N$",
                 ylabel="Avg. sum-rate (b/s/Hz)", out_dir=fig_dir, name="scal_sumrate_vs_N")

# (2) QoS vs N
qos_curves = {lab: {"mean": [scal_results[N][lab]["qos_mean"] for N in N_LIST],
                    "ci":   [scal_results[N][lab]["qos_ci"]   for N in N_LIST]} for lab in labels}
plot_metric_vs_x(N_LIST, qos_curves, xlabel="STAR-RIS elements $N$",
                 ylabel="QoS satisfaction probability", out_dir=fig_dir, name="scal_qos_vs_N")

display(Image(filename=os.path.join(fig_dir, "scal_sumrate_vs_N.png")))
display(Image(filename=os.path.join(fig_dir, "scal_qos_vs_N.png")))

# (3) Decomposition gain = SR(MADDPG) - SR(best single-agent) theo N
if "MADDPG" in labels and len(labels) > 1:
    single = [l for l in labels if l != "MADDPG"]
    gains, gain_labels = [], []
    for N in N_LIST:
        sr_madd = scal_results[N]["MADDPG"]["sr_mean"]
        best_single = max(scal_results[N][l]["sr_mean"] for l in single)
        gains.append(sr_madd - best_single); gain_labels.append(str(N))
    plt.figure(figsize=(5.8, 3.7))
    plt.axhline(0.0, color="gray", lw=0.8, ls="--")
    plt.plot(N_LIST, gains, marker="o", color="#1f77b4", lw=2)
    plt.fill_between(N_LIST, 0, gains, alpha=0.15, color="#1f77b4")
    plt.xlabel("STAR-RIS elements $N$")
    plt.ylabel("Decomposition gain\nSR(MADDPG) − SR(best single-agent)")
    plt.title("Lợi thế MADDPG tăng theo N (kỳ vọng đi lên)")
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "scal_decomposition_gain.png"), dpi=150)
    plt.show()

    print("\n--- Decomposition gain theo N ---")
    for N, g in zip(N_LIST, gains):
        flag = "✅ MADDPG dẫn" if g > 0 else "⚠️ chưa dẫn"
        print(f"  N={N:3d}: gain = {g:+.3f} b/s/Hz   {flag}")
    if len(gains) >= 2 and gains[-1] > gains[0]:
        print(f"\n  ✅ XU HƯỚNG ĐÚNG: gain tăng từ {gains[0]:+.3f} (N={N_LIST[0]}) -> {gains[-1]:+.3f} (N={N_LIST[-1]})")
        print("     => Củng cố lý thuyết: lợi thế phân rã CTDE tăng theo không gian hành động.")
    else:
        print("\n  ⚠️ Gain CHƯA tăng theo N. Cân nhắc: thêm N lớn hơn (96/128), hoặc nhấn thế mạnh khác"
              " (latency, decentralized execution).")

### 6. Diễn giải cho bài báo

- **Nếu gain tăng theo N** → đây là *đóng góp định lượng* mạnh nhất cho việc chọn MADDPG:
  *"Khi N tăng (không gian hành động RIS phình to), MADDPG vượt dần đơn-agent nhờ phân rã CTDE."*
  Đặt cạnh kết quả latency (MADDPG ~13× nhanh hơn BCD) → câu chuyện hoàn chỉnh: **gần tối ưu, độ trễ thấp,
  và mở rộng tốt theo quy mô RIS.**
- **Nếu gain phẳng/âm** → trung thực: ở dải N này MADDPG ngang đơn-agent; điểm bán là **thực thi phi tập trung
  (decentralized execution)** + latency, không phải sum-rate. Cân nhắc N lớn hơn để lộ xu hướng.
